# Seaborn E-Commerce Data Analysis

**Dataset:** `messy_ecommerce_15000_student_practice.csv`  
**Output:** `cleaned_ecommerce_dataset.csv`

This notebook follows the student task in order. Each task has its own code section, with short comments only where they help explain the code.

> **Important:** The original dataset is loaded only for analysis. It is not overwritten.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Spectral")
plt.rcParams["figure.facecolor"] = "#f8f9fa"

INPUT_FILE = "messy_ecommerce_15000_student_practice.csv"
CLEAN_FILE = "cleaned_ecommerce_dataset.csv"

df = pd.read_csv(INPUT_FILE)

print("Dataset shape:", df.shape)
display(df.head())


## Task 1 — Analyze Numerical Columns

In [ ]:
numerical_cols = df.select_dtypes(include="number").columns.tolist()
print("Numerical columns:")
print(numerical_cols)

display(df[numerical_cols].describe())


In [ ]:
# Requested descriptive statistics, including the 25th, 50th and 75th percentiles.
summary = df[numerical_cols].agg(
    ["mean", "median", "std", "min", "max"]
).T

percentiles = df[numerical_cols].quantile([0.25, 0.50, 0.75]).T
percentiles.columns = ["25%", "50%", "75%"]

summary = summary.join(percentiles)
display(summary)


In [ ]:
range_table = pd.DataFrame({
    "Minimum": df[numerical_cols].min(),
    "Maximum": df[numerical_cols].max(),
})
range_table["Range"] = range_table["Maximum"] - range_table["Minimum"]
display(range_table.sort_values("Range", ascending=False))


### Observations
- `Unit_Price` and `Total_Amount` have unusually large minimum-to-maximum ranges.
- `Age`, `Rating`, `Discount`, and `Delivery_Days` are on relatively smaller scales.
- `Total_Amount` is strongly right-skewed because a small number of orders are much larger than typical orders.
- `Quantity` is concentrated around 1–2 units, while its maximum is 5.
- The very large `Unit_Price` and `Total_Amount` values should be investigated as possible extreme business values or data-entry issues.


## Task 2 — Analyze Categorical Columns

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print("Categorical columns:")
print(categorical_cols)

display(df[categorical_cols].describe().T)


In [ ]:
categorical_summary = pd.DataFrame({
    "Unique_Values": df[categorical_cols].nunique(dropna=True),
    "Most_Frequent": [df[c].mode(dropna=True).iloc[0] for c in categorical_cols],
    "Frequency": [df[c].value_counts(dropna=True).iloc[0] for c in categorical_cols]
}, index=categorical_cols)

display(categorical_summary)


In [ ]:
for col in ["Gender", "City", "Product_Category", "Payment_Method", "Returned"]:
    print(f"\nMost common {col}:")
    print(df[col].value_counts(dropna=False).head(10))


### Observation
The most common categories can be read directly from the frequency tables above. Missing values are also visible because `dropna=False` is used.


## Task 3 — Find Unique Values

In [ ]:
for col in ["Gender", "City", "Product_Category", "Payment_Method", "Returned"]:
    print(f"\n{col} unique values:")
    print(df[col].dropna().unique())
    print("Number of unique values:", df[col].nunique(dropna=True))


### Observations
- `Gender` contains case variants such as `Male`/`male` and `Female`/`female`.
- `Payment_Method` contains `UPI`/`upi` and `Credit Card`/`credit card`.
- `Returned` contains `Yes`/`yes`.
- `Other` is a valid-looking gender category rather than a case inconsistency.
- The remaining category values should be checked for business validity before analysis.


## Task 4 — Identify and Standardize Inconsistent Categorical Data

In [ ]:
# Show the case-sensitive values that need standardization.
for col in ["Gender", "Payment_Method", "Returned"]:
    print(f"\n{col}:")
    print(sorted(df[col].dropna().astype(str).unique()))


In [ ]:
clean_df = df.copy()

clean_df["Gender"] = clean_df["Gender"].astype("string").str.strip().str.title()
clean_df["Payment_Method"] = clean_df["Payment_Method"].astype("string").str.strip().str.title()
clean_df["Returned"] = clean_df["Returned"].astype("string").str.strip().str.title()

for col in ["Gender", "Payment_Method", "Returned"]:
    print(f"\nStandardized {col}:")
    print(sorted(clean_df[col].dropna().unique()))


In [ ]:
# Verify the known duplicate representations are gone.
checks = {
    "male": clean_df["Gender"].eq("male").sum(),
    "female": clean_df["Gender"].eq("female").sum(),
    "upi": clean_df["Payment_Method"].eq("upi").sum(),
    "credit card": clean_df["Payment_Method"].eq("credit card").sum(),
    "yes": clean_df["Returned"].eq("yes").sum(),
}
print(checks)


### Observation
The inconsistent lowercase forms are converted to a common representation using `str.title()`. The verification cell should return zero for the lowercase checks.


## Task 5 — Handle Missing Values

In [ ]:
missing_before = clean_df.isna().sum().sort_values(ascending=False)
display(missing_before[missing_before > 0])


### Cleaning decisions
- **Age:** median — numeric and potentially affected by extreme ages.
- **Discount:** median — preserves the typical discount without being influenced strongly by extremes.
- **Rating:** median — keeps the numeric scale and avoids losing records.
- **Delivery_Days:** median — appropriate for a numeric operational measure.
- **Gender, City, Payment_Method:** mode — these are categorical variables, so the most common valid category is appropriate.
- Columns with no missing values are left unchanged.


In [ ]:
numeric_fill = ["Age", "Discount", "Rating", "Delivery_Days"]
categorical_fill = ["Gender", "City", "Payment_Method"]

for col in numeric_fill:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in categorical_fill:
    clean_df[col] = clean_df[col].fillna(clean_df[col].mode()[0])

print("Missing values after treatment:")
display(clean_df.isna().sum())


## Task 6 — Verify Missing Values

In [ ]:
missing_count = clean_df.isna().sum()
missing_pct = (missing_count / len(clean_df) * 100).round(2)

missing_report = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_pct
})
display(missing_report[missing_report["Missing_Count"] > 0])

print("Total missing values:", clean_df.isna().sum().sum())


In [ ]:
print("Dimensions before missing-value treatment:", df.shape)
print("Dimensions after missing-value treatment:", clean_df.shape)


### Observation
Missing-value treatment does not remove records because the selected methods replace missing values. Therefore, the row count remains unchanged at this stage.


## Task 7 — Detect Duplicate Records

In [ ]:
duplicate_count = clean_df.duplicated().sum()
print("Total exact duplicate rows:", duplicate_count)

duplicate_rows = clean_df[clean_df.duplicated(keep=False)].sort_values("Order_ID")
display(duplicate_rows)


In [ ]:
print("Duplicate Order_ID values:", clean_df["Order_ID"].duplicated().sum())
print("Number of unique Order_ID values:", clean_df["Order_ID"].nunique())

duplicate_groups = duplicate_rows.groupby("Order_ID")
exact_duplicate_groups = sum(len(group.drop_duplicates()) == 1 for _, group in duplicate_groups)

print("Duplicate Order_ID groups:", len(duplicate_groups))
print("Groups that are exact duplicates:", exact_duplicate_groups)


### Observation
Duplicate rows can inflate order counts, sales totals, category frequencies, and averages. In this dataset the duplicated `Order_ID` groups are exact duplicate records, so removing the exact duplicates is justified.


## Task 7A — Remove Duplicate Records

In [ ]:
records_before = len(clean_df)
duplicates_before = clean_df.duplicated().sum()

clean_df = clean_df.drop_duplicates().reset_index(drop=True)

records_after = len(clean_df)

print("Records before:", records_before)
print("Duplicate records removed:", duplicates_before)
print("Records after:", records_after)
print("Remaining exact duplicates:", clean_df.duplicated().sum())
print("New dimensions:", clean_df.shape)


## Task 8 — Detect Outliers Using Seaborn

In [ ]:
outlier_cols = [
    "Age", "Unit_Price", "Quantity", "Discount",
    "Rating", "Delivery_Days", "Total_Amount"
]

for col in outlier_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=clean_df[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.show()


### Observations
- `Unit_Price` and `Total_Amount` show the strongest extreme-value behavior.
- `Quantity` also has upper-tail values because most orders contain only a small number of units.
- `Age`, `Rating`, and `Delivery_Days` have fewer extreme values.
- An outlier is not automatically an error. A high-value order can be a legitimate transaction.
- Extreme `Unit_Price` and `Total_Amount` records deserve further investigation because they can have a large effect on means and correlations.


## Task 9 — Detect Outliers Using IQR

In [ ]:
def iqr_report(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)

    report = {
        "Column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(mask.sum())
    }
    return report, data.loc[mask]

iqr_reports = []
outlier_records = {}

for col in ["Age", "Unit_Price", "Quantity"]:
    report, records = iqr_report(clean_df, col)
    iqr_reports.append(report)
    outlier_records[col] = records

display(pd.DataFrame(iqr_reports))

for col, records in outlier_records.items():
    print(f"\nFirst outlier records for {col}:")
    display(records[["Order_ID", col]].head(10))


### Observation
The IQR method identifies statistical candidates for investigation. A large number of IQR outliers does not prove that the values are wrong, especially for naturally right-skewed business variables such as price and order value.


## Task 10 — Outlier Decision

In [ ]:
# Inspect the most extreme business values rather than deleting all IQR outliers.
extreme_values = clean_df.nlargest(10, "Unit_Price")[
    ["Order_ID", "Product", "Product_Category", "Unit_Price", "Quantity", "Discount", "Total_Amount"]
]
display(extreme_values)

extreme_orders = clean_df.nlargest(10, "Total_Amount")[
    ["Order_ID", "Product", "Product_Category", "Unit_Price", "Quantity", "Discount", "Total_Amount"]
]
display(extreme_orders)


### Decision
No outlier is automatically removed in this notebook. The IQR method is used for **detection**, not automatic deletion.

- Values that are plausible for a real product/order are treated as **legitimate extreme values**.
- Values that look impossible or contradict business rules should be marked **suspicious** and investigated.
- A value should be removed only after a clear data-quality rule or source verification supports removal.


## Task 11 — Data Consistency Validation

In [ ]:
# Discount is treated as a percentage.
clean_df["Expected_Total"] = (
    clean_df["Unit_Price"]
    * clean_df["Quantity"]
    * (100 - clean_df["Discount"]) / 100
)

clean_df["Amount_Difference"] = (
    clean_df["Expected_Total"] - clean_df["Total_Amount"]
).abs()

display(clean_df[
    ["Order_ID", "Unit_Price", "Quantity", "Discount",
     "Expected_Total", "Total_Amount", "Amount_Difference"]
].head())


In [ ]:
print("Difference summary:")
display(clean_df["Amount_Difference"].describe())

large_difference = clean_df[clean_df["Amount_Difference"] > 1].sort_values(
    "Amount_Difference", ascending=False
)

print("Records with difference greater than ₹1:", len(large_difference))
display(large_difference[
    ["Order_ID", "Unit_Price", "Quantity", "Discount",
     "Expected_Total", "Total_Amount", "Amount_Difference"]
].head(20))


### Observation
The calculated amount should normally differ from `Total_Amount` only by small rounding differences because the stored total has two decimal places. A difference greater than ₹1 is a useful investigation threshold.


## Task 12 — Final Data Quality Check

In [ ]:
# Convert the date only after checking its validity.
clean_df["Order_Date"] = pd.to_datetime(clean_df["Order_Date"], errors="coerce")

print("Dimensions:", clean_df.shape)
print("\nData types:")
print(clean_df.dtypes)

print("\nMissing values:", clean_df.isna().sum().sum())
print("Exact duplicates:", clean_df.duplicated().sum())

print("\nDate parsing failures:", clean_df["Order_Date"].isna().sum())

print("\nUnique values:")
for col in ["Gender", "City", "Product_Category", "Payment_Method", "Returned"]:
    print(col, "=", clean_df[col].nunique())

print("\nCategorical values:")
for col in ["Gender", "Payment_Method", "Returned"]:
    print(col, sorted(clean_df[col].dropna().unique()))


In [ ]:
print("Statistical summary:")
display(clean_df.describe(include="all").T)


### Final quality conclusion
The dataset is suitable for exploratory data analysis after:
1. standardizing categorical case differences,
2. treating missing values column by column,
3. removing exact duplicate records,
4. validating dates,
5. checking IQR-based outliers without blindly deleting them, and
6. validating `Total_Amount` against the expected order amount.

The remaining extreme values should be interpreted carefully because they can influence averages and correlations.


## Task 13 — Save the Clean Dataset

In [ ]:
# Expected_Total and Amount_Difference are validation columns; they are not part of the original schema.
clean_output = clean_df.drop(columns=["Expected_Total", "Amount_Difference"])

clean_output.to_csv(CLEAN_FILE, index=False)

print("Saved:", CLEAN_FILE)
print("Clean dataset shape:", clean_output.shape)


## Task 14 — Seaborn Univariate Analysis

In [ ]:
# 1. Histogram of Age
plt.figure(figsize=(8, 4))
sns.histplot(data=clean_output, x='Age', bins=20, kde=True, color='deeppink')
plt.title("Age Distribution")
plt.show()


**Observation:** Most customers are concentrated around the adult age range, with relatively few observations at the extreme ages.

In [ ]:
# 2. Histogram of Total_Amount
plt.figure(figsize=(8, 4))
sns.histplot(data=clean_output, x='Total_Amount', bins=40, color='orange')
plt.title("Total Amount Distribution")
plt.show()


**Observation:** `Total_Amount` is strongly right-skewed, with a long upper tail caused by a relatively small number of high-value orders.

In [ ]:
# 3. KDE plot of Unit_Price
plt.figure(figsize=(8, 4))
sns.kdeplot(data=clean_output, x='Unit_Price', fill=True, color='mediumseagreen')
plt.title("Unit Price Distribution")
plt.show()


**Observation:** The unit-price distribution is concentrated at lower values with a long right tail, indicating a small number of expensive products.

In [ ]:
# 4. Boxplot of Total_Amount
plt.figure(figsize=(8, 4))
sns.boxplot(data=clean_output, x='Total_Amount', color='violet')
plt.title("Total Amount Boxplot")
plt.show()


**Observation:** The boxplot confirms many high-value observations beyond the upper whisker; these should be investigated rather than automatically deleted.

In [ ]:
# 5. Countplot of Product_Category
plt.figure(figsize=(9, 4))
sns.countplot(data=clean_output, x="Product_Category", order=clean_output["Product_Category"].value_counts().index)
plt.xticks(rotation=30)
plt.title("Orders by Product Category")
plt.show()


**Observation:** Electronics has the highest order count among the product categories.

In [ ]:
# 6. Countplot of Payment_Method
plt.figure(figsize=(8, 4))
sns.countplot(data=clean_output, x="Payment_Method", order=clean_output["Payment_Method"].value_counts().index)
plt.xticks(rotation=30)
plt.title("Orders by Payment Method")
plt.show()


**Observation:** Credit Card is the most frequently used payment method after standardizing the category labels.

In [ ]:
# 7. Countplot of Gender
plt.figure(figsize=(7, 4))
sns.countplot(data=clean_output, x="Gender", order=clean_output["Gender"].value_counts().index)
plt.title("Orders by Gender")
plt.show()


**Observation:** Female customers account for the largest number of orders, followed by male customers.

## Task 15 — Seaborn Bivariate Analysis

In [ ]:
# Age vs Total_Amount
plt.figure(figsize=(8, 5))
sns.scatterplot(data=clean_output, x="Age", y="Total_Amount", alpha=0.5)
plt.title("Age vs Total Amount")
plt.show()


**Interpretation:** There is no strong visible linear relationship between age and total order value; high-value orders occur across different ages.

In [ ]:
# Unit_Price vs Total_Amount
plt.figure(figsize=(8, 5))
sns.scatterplot(data=clean_output, x="Unit_Price", y="Total_Amount", alpha=0.5)
plt.title("Unit Price vs Total Amount")
plt.show()


**Interpretation:** Total order value generally increases with unit price, which is expected because unit price directly contributes to the order amount.

In [ ]:
# Discount vs Total_Amount
plt.figure(figsize=(8, 5))
sns.scatterplot(data=clean_output, x="Discount", y="Total_Amount", alpha=0.5)
plt.title("Discount vs Total Amount")
plt.show()


**Interpretation:** Discount has only a weak relationship with total order value in this dataset.

In [ ]:
# Product Category vs Total Amount
plt.figure(figsize=(9, 5))
sns.boxplot(data=clean_output, x="Product_Category", y="Total_Amount")
plt.xticks(rotation=30)
plt.title("Product Category vs Total Amount")
plt.show()


**Interpretation:** Electronics has a much higher order-value distribution than the other categories, although its distribution also contains extreme values.

In [ ]:
# Gender vs Total Amount
plt.figure(figsize=(7, 5))
sns.boxplot(data=clean_output, x="Gender", y="Total_Amount")
plt.title("Gender vs Total Amount")
plt.show()


**Interpretation:** The spending distributions overlap substantially across genders, so gender alone is not a strong predictor of order value.

In [ ]:
# Payment Method vs Total Amount
plt.figure(figsize=(9, 5))
sns.boxplot(data=clean_output, x="Payment_Method", y="Total_Amount")
plt.xticks(rotation=30)
plt.title("Payment Method vs Total Amount")
plt.show()


**Interpretation:** Order values vary across payment methods, but the distributions overlap, so payment method alone does not explain most variation in total amount.

## Task 16 — Multivariate Analysis

In [ ]:
selected_num = [
    "Age", "Quantity", "Unit_Price",
    "Discount", "Rating", "Delivery_Days", "Total_Amount"
]

sns.pairplot(clean_output[selected_num], corner=True)
plt.show()


**Pairplot observation:** `Unit_Price` and `Total_Amount` show the clearest positive relationship. Most other numerical pairs have weak visible relationships.

In [ ]:
plt.figure(figsize=(10, 7))
corr = clean_output[selected_num].corr()

sns.heatmap(corr, annot=True, fmt='.2f', cmap='turbo', center=0)
plt.title("Correlation Heatmap")
plt.show()


**Heatmap observation:** Unit price has the strongest positive relationship with total amount. Discount has the strongest negative relationship, but it is still weak.

In [ ]:
# Categorical comparison using hue
plt.figure(figsize=(9, 5))
sns.countplot(
    data=clean_output,
    x="Product_Category",
    hue="Gender"
)
plt.xticks(rotation=30)
plt.title("Product Category by Gender")
plt.show()


**Interpretation:** The category counts differ by gender, while Electronics remains the largest category across the major gender groups.

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(
    data=clean_output,
    x="Product_Category",
    hue="Returned"
)
plt.xticks(rotation=30)
plt.title("Product Category by Return Status")
plt.show()


**Interpretation:** Most orders are not returned in every product category, while return counts vary with category volume.

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(
    data=clean_output,
    x="Payment_Method",
    hue="Returned"
)
plt.xticks(rotation=30)
plt.title("Payment Method by Return Status")
plt.show()


**Interpretation:** Non-returned orders dominate across all payment methods; return behavior is not determined by payment method alone.

## Task 17 — Correlation Analysis

In [ ]:
correlation_matrix = clean_output[selected_num].corr()
display(correlation_matrix)

target_corr = correlation_matrix["Total_Amount"].drop("Total_Amount").sort_values(ascending=False)

print("Strongest positive relationship with Total_Amount:")
print(target_corr.iloc[0])

print("\nStrongest negative relationship with Total_Amount:")
print(target_corr.sort_values().iloc[0])

print("\nWeak relationships with Total_Amount (absolute correlation < 0.10):")
display(target_corr[abs(target_corr) < 0.10])


### Interpretation
- **Strongest positive correlation:** `Unit_Price` and `Total_Amount`.
- **Strongest negative correlation:** `Discount` has the most negative relationship, but it is weak.
- **Weak relationships:** Age, Rating, Delivery_Days, and Discount are weakly correlated with Total_Amount.
- **Useful features for predicting Total_Amount:** Unit_Price is clearly the strongest numerical feature, followed by Quantity.


## Task 18 — Business Insights

In [ ]:
category_orders = clean_output["Product_Category"].value_counts()
category_avg = clean_output.groupby("Product_Category")["Total_Amount"].mean().sort_values(ascending=False)
city_sales = clean_output.groupby("City")["Total_Amount"].sum().sort_values(ascending=False)
gender_avg = clean_output.groupby("Gender")["Total_Amount"].mean().sort_values(ascending=False)
return_pct = clean_output["Returned"].eq("Yes").mean() * 100
product_rating = clean_output.groupby("Product")["Rating"].mean().sort_values(ascending=False)
discount_corr = clean_output[["Discount", "Total_Amount"]].corr().iloc[0, 1]
delivery_rating_corr = clean_output[["Delivery_Days", "Rating"]].corr().iloc[0, 1]

print("1. Most orders:", category_orders.index[0], "-", category_orders.iloc[0])
print("2. Highest average order value category:", category_avg.index[0], "-", round(category_avg.iloc[0], 2))
print("3. Most popular payment method:", clean_output["Payment_Method"].value_counts().idxmax())
print("4. City with highest sales:", city_sales.index[0], "-", round(city_sales.iloc[0], 2))
print("5. Gender with highest average spending:", gender_avg.index[0], "-", round(gender_avg.iloc[0], 2))
print("6. Discount vs Total Amount correlation:", round(discount_corr, 3))
print("7. Highest average-rated product:", product_rating.index[0], "-", round(product_rating.iloc[0], 2))
print("8. Returned-order percentage:", round(return_pct, 2), "%")
print("9. Delivery Days vs Rating correlation:", round(delivery_rating_corr, 3))
print("10. Strongest numerical feature for Total Amount:", "Unit_Price")


### Business interpretation
1. Electronics receives the most orders.
2. Electronics also has the highest average order value.
3. Credit Card is the most popular payment method after category standardization.
4. Kolkata has the highest total sales in the cleaned dataset.
5. The `Other` gender group has the highest average spending, but it has far fewer orders, so this should be interpreted cautiously.
6. Discount has only a weak negative relationship with total amount.
7. Hair Dryer has the highest average product rating in this dataset.
8. About 12% of orders are returned.
9. Delivery time and rating have almost no linear relationship.
10. Unit price is the strongest numerical factor associated with total order value.


## Final Submission Checklist

In [ ]:
print("Notebook analysis complete.")
print("Clean dataset file:", CLEAN_FILE)
print("Original dataset remains:", INPUT_FILE)
print("Final cleaned dimensions:", clean_output.shape)


### Conclusion
The cleaned dataset is ready for EDA because major missing-value, duplicate, categorical-consistency, date-format, and amount-consistency issues have been addressed. Statistical outliers remain flagged for interpretation rather than being blindly removed, which preserves potentially legitimate business transactions.
